In [2]:
!pip install ollama

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
from pathlib import Path
import ollama

modelo = "phi4-mini"

prompt_1 = Path("prompt_1.md").read_text(encoding="utf-8")
prompt_2 = Path("prompt_2.md").read_text(encoding="utf-8")
prompt_3 = Path("prompt_3.md").read_text(encoding="utf-8")

In [8]:
def procesar_mensaje(mensaje_usuario: str, temperature: float = 1.0) -> dict:
    respuesta = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": prompt_1},
            {"role": "user", "content": mensaje_usuario},
        ],
        format="json",
        options={
            "temperature": temperature,
        },
    )

    output_1 = json.loads(respuesta["message"]["content"])

    respuesta = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": prompt_2},
            {"role": "user", "content": json.dumps(output_1, ensure_ascii=False)},
        ],
        format="json",
        options={
            "temperature": temperature,
        },
    )

    output_2 = json.loads(respuesta["message"]["content"])

    respuesta = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": prompt_3},
            {"role": "user", "content": json.dumps(output_2, ensure_ascii=False)},
        ],
        format="json",
        options={
            "temperature": temperature,
        },
    )

    return json.loads(respuesta["message"]["content"])

In [5]:
mensajes_prueba = [
    "Compré una campera y me llegó con una mancha y una costura rota. Para colmo, revisé el resumen y me la cobraron en dos cuotas cuando elegí pago único. Quiero un cambio inmediato y que me solucionen el tema de la tarjeta.",
    "Mi pedido tardó dos semanas más de lo avisado y cuando abrió la caja faltaban la mitad de las cosas. Les escribí por el chat y nadie me contestó en todo el día. Devuélvanme la plata de lo que no llegó o hago el reclamo.",
    "El teléfono que compré se apaga solo a cada rato y no carga bien. Además me vino un cargo extra por un seguro que nunca pedí. Arreglen esto o les doy de baja la compra.",
    "La zapatilla izquierda vino despegada y encima me cobraron el envío que decía 'gratis' en la web. Exijo que me manden otro par sin cargo y que me devuelvan el dinero del envío ya.",
    "El ventilador que compré hace una semana hace un ruido horrible y no gira. Fui a la sucursal a reclamar y el empleado me trató re mal. Quiero que me devuelvan el dinero en efectivo o los denuncio."
]

In [9]:
mensaje_experimento = mensajes_prueba[0]
print(
    "Mensaje seleccionado para el"
    f' experimento:\n"{mensaje_experimento}"\n'
)

resultado = procesar_mensaje(mensaje_experimento, 0)
print(resultado)

Mensaje seleccionado para el experimento:
"Compré una campera y me llegó con una mancha y una costura rota. Para colmo, revisé el resumen y me la cobraron en dos cuotas cuando elegí pago único. Quiero un cambio inmediato y que me solucionen el tema de la tarjeta."

{'metadatos_clasificacion': {'categoria': 'Producto Defectuoso', 'nivel_urgencia': 'Alta'}, 'analisis_interno': {'cadena_de_pensamiento': '1. El cliente informa un producto defectuoso con una mancha y una costura rota. 2. El problema de facturación se refiere a una tarifa incorrecta aplicada al pago. 3. La política de producto defectuoso con más de 30 días desde la compra indica crear un ticket. La política de problema de facturación también requiere crear un ticket. Por haber más de una política aplicable, corresponde usar ACC_CREAR_TICKET.', 'politica_aplicada': 'Producto defectuoso con más de 30 días desde la compra: crear un ticket; problema de facturación: crear un ticket. Por haber más de una política aplicable, corres

### Input:

> "Compré una campera y me llegó con una mancha y una costura rota. Para colmo, revisé el resumen y me la cobraron en dos cuotas cuando elegí pago único. Quiero un cambio inmediato y que me solucionen el tema de la tarjeta."


``` json
{
  "metadatos_clasificacion": {
    "categoria": "Producto Defectuoso",
    "nivel_urgencia": "Alta"
  },
  "analisis_interno": {
    "cadena_de_pensamiento": "1. El cliente informa un producto defectuoso con una mancha y una costura rota. 2. El problema de facturación se refiere a una tarifa incorrecta aplicada al pago. 3. La política de producto defectuoso con más de 30 días desde la compra indica crear un ticket. La política de problema de facturación también requiere crear un ticket. Por haber más de una política aplicable, corresponde usar ACC_CREAR_TICKET.",
    "politica_aplicada": "Producto defectuoso con más de 30 días desde la compra: crear un ticket; problema de facturación: crear un ticket. Por haber más de una política aplicable, corresponde crear un ticket."
  },
  "accion_y_respuesta": {
    "codigo_accion_sistema": "ACC_CREAR_TICKET",
    "mensaje_al_cliente": "Lamentamos los inconvenientes ocasionados. Registraremos dos reclamos: uno para el producto defectuoso con una mancha y una costura rota, y otro para la tarifa incorrecta aplicada al pago. El área correspondiente revisará ambos casos y le comunicará la resolución una vez finalizado el análisis."
  }
}
```

En el modelo phi4-mini de ollama el tiempo de respuesta fue más lento (5 min) y la precisión fue menor, lo cual es razonable ya que elegi un modelo con menor cantidad de parametros. No pudo identificar la politica de falta de informacion para los productos defectuosos y ademas la categoria deberia ser facturacion al tener mayor priolidad.